# AModule 1 Part 2: Agents

In [1]:
import os
import json
from dotenv import dotenv_values

from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from sqlitesearch import TextSearchIndex

from openai import OpenAI

In [2]:
secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=api_key)

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
index.search('how do I run ollama?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

## Without tools...

In [5]:
message_history = [
    {"role": "user", "content": "Can I still join the course?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
)

response.output_text

'Possibly — it depends on the course’s enrollment deadline and whether there are still spots open.\n\nIf you want, send me:\n- the course name,\n- the school/platform,\n- and whether you mean late enrollment or waitlist,\n\nand I can help you figure out the next step or draft a message to the instructor/registrar.'

In [6]:
response.output

[ResponseOutputMessage(id='msg_0016a8075e017d7b006a30efa1351c8191b8cf8c1f7abd15dc', content=[ResponseOutputText(annotations=[], text='Possibly — it depends on the course’s enrollment deadline and whether there are still spots open.\n\nIf you want, send me:\n- the course name,\n- the school/platform,\n- and whether you mean late enrollment or waitlist,\n\nand I can help you figure out the next step or draft a message to the instructor/registrar.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

## With tools...

First we define the tool. We do this in a language agnostic way, i.e. we always use JSON.

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment add drop deadline registration"}', call_id='call_wAuw5Ai5b3ChlFa5ghkCwm9S', name='search', type='function_call', id='fc_064209e234f7f0cc006a30efa34c2881a3b31d167fdfc2fc9f', namespace=None, status='completed')]

Instead of returning a `ResponseOutputMessage` we return a `ResponseFunctionToolCall`

### Executing the function and sending back the result

In [10]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment add drop deadline registration"}', call_id='call_wAuw5Ai5b3ChlFa5ghkCwm9S', name='search', type='function_call', id='fc_064209e234f7f0cc006a30efa34c2881a3b31d167fdfc2fc9f', namespace=None, status='completed')

In [11]:
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
message_history.extend(response.output)

In [13]:
message_history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
    })

In [14]:
message_history

[{'role': 'user', 'content': 'Can I still join the course?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment add drop deadline registration"}', call_id='call_wAuw5Ai5b3ChlFa5ghkCwm9S', name='search', type='function_call', id='fc_064209e234f7f0cc006a30efa34c2881a3b31d167fdfc2fc9f', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_wAuw5Ai5b3ChlFa5ghkCwm9S',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Ye

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, you need to submit your project while submissions are still open.'

In [16]:
response.usage

ResponseUsage(input_tokens=613, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=31, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=644)

In [17]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005992500000000001

## The Agentic Loop

The agent loop will be formed of 3 main parts:
- Instructions
- Tools
- Memory

In [18]:
# The starting point - instructions

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "Can I still join the course?"

message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [26]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history,
    tools=[search_tool]
)



In [21]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course still join enrollment late registration deadline course FAQ"}', call_id='call_LZnDW2Z7FERy0XKQxFoMnrNz', name='search', type='function_call', id='fc_0130dd2f0a0039b6006a30f040a77c8191ae2c337575de08e5', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course course FAQ enrollment access late join"}', call_id='call_hxdtDQylHg4cZw0UxBqSkZo4', name='search', type='function_call', id='fc_0130dd2f0a0039b6006a30f040a7a481919e9e1d1bedd2b4dc', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course start late registration can I still join FAQ"}', call_id='call_cJy7WkNB4ugHw6QQAYaKNoEu', name='search', type='function_call', id='fc_0130dd2f0a0039b6006a30f040a7ac8191a23ddfb693e44054', namespace=None, status='completed')]

We can create a make call function to take the actions that we completed for a single function call:
- 

In [23]:
def make_call(call):
    """
    Function to make tool call and return result to llm.
    - call: List[ResponseOutputItem]
    Returns dict to be added to message history
    """
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [30]:
message_history.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function:', item.name, item.arguments)
        function_call_output = make_call(item)
        message_history.append(function_call_output)

    elif item.type == 'message':
        print(item.content[0].text)

Yes, you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. If you just want to learn, you can start even if you joined late.

If you want, I can also tell you about certificate requirements or whether you need to register first.


In [31]:
message_history

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'Can I still join the course?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course still join enrollment late registration deadline course FAQ"}', call_id='call_LZnDW2Z7FERy0XKQxFoMnrNz', name='search', type='function_call', id='fc_0130dd2f0a0039b6006a30f040a77c8191ae2c337575de08e5', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course course FAQ enrollment access late join"}', call_id='call

In [32]:
message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:

    print(f"iteration #{it}...")

    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history,
        tools=[search_tool]
    )

    message_history.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function:', item.name, item.arguments)
            function_call_output = make_call(item)
            message_history.append(function_call_output)
            has_function_calls = True

        elif item.type == 'message':
            print(item.content[0].text)

    it += 1
    if has_function_calls == False:
        break

iteration #1...
function: search {"query":"Can I still join the course enrollment late add drop deadline join course late registration"}
function: search {"query":"course join late enrollment waitlist add drop FAQ"}
iteration #2...
Yes — you can still join the course.

If you want a certificate, though, you’ll need to submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.

Do you want to know more about certificates, homework, or when the course runs next?


In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform a search, analyse the results and then perform additional searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [34]:
message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:

    print(f"iteration #{it}...")

    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history,
        tools=[search_tool]
    )

    message_history.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function:', item.name, item.arguments)
            function_call_output = make_call(item)
            message_history.append(function_call_output)
            has_function_calls = True

        elif item.type == 'message':
            print(item.content[0].text)

    it += 1
    if has_function_calls == False:
        break

iteration #1...
function: search {"query":"join the course enrollment deadline late registration can I still join the course"}
iteration #2...
function: search {"query":"course still join accepted start learning submitting homework while form is open registration gauge interest self-paced certificate live cohort"}
iteration #3...
Yes — you can still join the course.

A couple of notes:
- You can start learning and submit homework as long as the submission form is open.
- If you want a certificate, you need to submit your project while submissions are still being accepted.
- The course is not checking a registered list, so registration isn’t required just to begin.

If you want, I can also explain how certificates and homework work in this course.


In [41]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function: search {"query":"Olama local run install run locally Ollama"}
iteration #2...
function: search {"query":"Ollama local run locally install model serve localhost FAQ"}
iteration #3...
I couldn’t find an FAQ entry for “run Ollama locally,” so here’s the general setup:

1. **Install Ollama**
   - Download it from the official Ollama website for your OS, or install via package manager if available.

2. **Start the local server**
   - Usually Ollama runs a local service automatically after installation.
   - You can check it with:
     ```bash
     ollama serve
     ```

3. **Run a model**
   - Pull and run a model, for example:
     ```bash
     ollama run llama3
     ```
   - This will download the model if needed and start an interactive chat locally.

4. **Use the local API**
   - Ollama exposes a local endpoint, typically on:
     ```text
     http://localhost:11434
     ```
   - You can call it from your code or tools.

5. **Verify it works**
   - Try:
     ``

'I couldn’t find an FAQ entry for “run Ollama locally,” so here’s the general setup:\n\n1. **Install Ollama**\n   - Download it from the official Ollama website for your OS, or install via package manager if available.\n\n2. **Start the local server**\n   - Usually Ollama runs a local service automatically after installation.\n   - You can check it with:\n     ```bash\n     ollama serve\n     ```\n\n3. **Run a model**\n   - Pull and run a model, for example:\n     ```bash\n     ollama run llama3\n     ```\n   - This will download the model if needed and start an interactive chat locally.\n\n4. **Use the local API**\n   - Ollama exposes a local endpoint, typically on:\n     ```text\n     http://localhost:11434\n     ```\n   - You can call it from your code or tools.\n\n5. **Verify it works**\n   - Try:\n     ```bash\n     ollama list\n     ollama ps\n     ```\n\nIf you want, I can also help with:\n- installing Ollama on Windows/Mac/Linux\n- running it with Docker\n- using it from Python

In [42]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function: search {"query":"queen gambit chess opening queen gambit what is it"}
iteration #2...
function: search {"query":"queen's gambit chess opening definition d4 d5 c4"}
iteration #3...
The Queen’s Gambit is a chess opening that starts with:

1. d4 d5  
2. c4

White offers the c-pawn as a “gambit” to try to gain control of the center. Black can accept the pawn or decline it.

Main idea:
- White wants strong central control and active piece play.
- Black decides whether to hold the pawn or focus on development.

Common responses:
- **Queen’s Gambit Accepted**: Black takes the c4 pawn.
- **Queen’s Gambit Declined**: Black keeps the pawn and supports d5.
- **Slav Defense**: Black supports d5 with ...c6.
- **Chigorin, Albin, etc.**: less common alternatives.

If you want, I can also explain:
- the difference between **accepted** and **declined**
- the basic plans for both sides
- famous Queen’s Gambit lines from the show and real chess


'The Queen’s Gambit is a chess opening that starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn as a “gambit” to try to gain control of the center. Black can accept the pawn or decline it.\n\nMain idea:\n- White wants strong central control and active piece play.\n- Black decides whether to hold the pawn or focus on development.\n\nCommon responses:\n- **Queen’s Gambit Accepted**: Black takes the c4 pawn.\n- **Queen’s Gambit Declined**: Black keeps the pawn and supports d5.\n- **Slav Defense**: Black supports d5 with ...c6.\n- **Chigorin, Albin, etc.**: less common alternatives.\n\nIf you want, I can also explain:\n- the difference between **accepted** and **declined**\n- the basic plans for both sides\n- famous Queen’s Gambit lines from the show and real chess'

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function: search {"query":"queen gambit course FAQ chess opening queen's gambit"}
iteration #2...
function: search {"query":"queen gambit queen's gambit FAQ course chess"}
iteration #3...
I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this isn’t a course/logistics question.

If you meant something else related to the course, feel free to rephrase it. Is there another course-related area you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this isn’t a course/logistics question.\n\nIf you meant something else related to the course, feel free to rephrase it. Is there another course-related area you want to explore?'

### Putting this into a function

In [48]:
def agent_loop(instructions, question, model='gpt-5.4-mini'):

    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    has_function_calls = True
    
    while has_function_calls:

        has_function_calls = False

        print(f"iteration #{it}...")

        response = openai_client.responses.create(
            model=model,
            input=message_history,
            tools=[search_tool]
        )

        message_history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function:', item.name, item.arguments)
                function_call_output = make_call(item)
                message_history.append(function_call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print(final_answer)
                
        it += 1

    return final_answer
    

In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform a search, analyse the results and then perform additional searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "Can I still join the course?"

agent_loop(instructions, question)

iteration #1...
function: search {"query":"Can I still join the course enroll late registration join course FAQ"}
iteration #2...
function: search {"query":"still join course certificate submit project while accepting submissions live cohort registration open FAQ"}
iteration #3...
Yes — you can still join the course.

If your goal is to get a certificate, you’ll need to submit your project while the course is still accepting submissions. Also, registration isn’t strictly required to start learning or submitting homework while the form is open.

If you want, I can also explain the certificate requirements or what to do if you missed earlier homework.


'Yes — you can still join the course.\n\nIf your goal is to get a certificate, you’ll need to submit your project while the course is still accepting submissions. Also, registration isn’t strictly required to start learning or submitting homework while the form is open.\n\nIf you want, I can also explain the certificate requirements or what to do if you missed earlier homework.'

In [50]:
# Cleaned agent loop without print statements

def clean_agent_loop(instructions, question, model='gpt-5.4-mini'):

    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    has_function_calls = True
    
    while has_function_calls:

        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=message_history,
            tools=[search_tool]
        )

        message_history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                function_call_output = make_call(item)
                message_history.append(function_call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                
        it += 1

    return final_answer

In [51]:
answer = clean_agent_loop(instructions, question)
print(answer)

Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Registration itself isn’t strictly required to start learning or submitting homework while the forms are open.

If you want, I can also tell you:
- whether you can still get a certificate if you join late, or
- what the requirements are for the capstone/project.


## Implementation with ToyAIKit framework